# Лабораторная работа №4

## Методы поиска условного экстремума

**Вариант 2**

Параметры функции Розенброка:

- `a = 150`
- `b = 2`
- `f0 = 100`
- `n = 3`

**Цель работы:**

1. Изучить алгоритмы условной оптимизации.
2. Реализовать методы поиска условного экстремума.
3. Найти оптимальное решение задачи с учетом ограничений.
4. Сравнить точность и скорость сходимости разных методов.

**Методы, реализуемые в работе:**

1. Метод штрафных функций.
2. Метод барьерных функций.
3. Комбинированный метод штрафных и барьерных функций.
4. Метод модифицированных функций Лагранжа.
5. Метод проекции градиента.


## 1. Импорт библиотек

В данном блоке подключаются библиотеки для численных вычислений, построения графиков, измерения времени работы алгоритмов и формирования итоговых таблиц.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

np.set_printoptions(precision=6, suppress=True)

## 2. Постановка задачи

Необходимо найти условный минимум функции Розенброка:

$$
f(x) = \sum_{i=1}^{n-1} \left[a(x_{i+1} - x_i^2)^2 + b(x_i - 1)^2\right] + f_0
$$

Для варианта 2:

$$
a = 150, \quad b = 2, \quad f_0 = 100, \quad n = 3
$$

Ограничения варианта 2:

$$
g_1(x) = x_1^2 + x_2^2 + x_3^2 - 1 \leq 0
$$

$$
g_2(x) = -x_1 \leq 0
$$

$$
g_3(x) = -x_2 \leq 0
$$

$$
g_4(x) = -x_3 \leq 0
$$

То есть допустимая область задается частью единичного шара в первом октанте:

$$
x_1 \geq 0, \quad x_2 \geq 0, \quad x_3 \geq 0, \quad x_1^2 + x_2^2 + x_3^2 \leq 1
$$


In [ ]:
a = 150
b = 2
f0 = 100
n = 3

# Внутренняя допустимая начальная точка
x0_feasible = np.array([0.2, 0.2, 0.2])

# Точка вне допустимой области для метода внешних штрафов
x0_external = np.array([-1.2, 1.0, 0.5])

eps = 1e-6
max_iter = 2000

print("Параметры варианта:")
print(f"a = {a}, b = {b}, f0 = {f0}, n = {n}")
print("Внутренняя начальная точка:", x0_feasible)
print("Внешняя начальная точка:", x0_external)

## 3. Функция Розенброка, градиент и ограничения

В данном блоке реализуются:

- целевая функция;
- градиент целевой функции;
- функции ограничений;
- проверка допустимости точки;
- штраф за нарушение ограничений.

Ограничение считается выполненным, если его значение меньше или равно нулю.

In [ ]:
def f(x):
    x = np.asarray(x, dtype=float)
    total = 0.0
    for i in range(len(x) - 1):
        total += a * (x[i + 1] - x[i] ** 2) ** 2 + b * (x[i] - 1) ** 2
    return total + f0


def grad_f(x):
    x = np.asarray(x, dtype=float)
    grad = np.zeros_like(x, dtype=float)

    for i in range(len(x) - 1):
        grad[i] += -4 * a * x[i] * (x[i + 1] - x[i] ** 2) + 2 * b * (x[i] - 1)
        grad[i + 1] += 2 * a * (x[i + 1] - x[i] ** 2)

    return grad


def constraints(x):
    x = np.asarray(x, dtype=float)
    return np.array([
        x[0] ** 2 + x[1] ** 2 + x[2] ** 2 - 1,
        -x[0],
        -x[1],
        -x[2]
    ])


def grad_constraints(x):
    x = np.asarray(x, dtype=float)
    return np.array([
        [2 * x[0], 2 * x[1], 2 * x[2]],
        [-1.0, 0.0, 0.0],
        [0.0, -1.0, 0.0],
        [0.0, 0.0, -1.0]
    ])


def is_feasible(x, tol=1e-8):
    return np.all(constraints(x) <= tol)


def violation_norm(x):
    g = constraints(x)
    return np.linalg.norm(np.maximum(g, 0.0))


print("f(x0_feasible) =", f(x0_feasible))
print("Ограничения в x0_feasible:", constraints(x0_feasible))
print("Допустима ли x0_feasible:", is_feasible(x0_feasible))

print("\nf(x0_external) =", f(x0_external))
print("Ограничения в x0_external:", constraints(x0_external))
print("Допустима ли x0_external:", is_feasible(x0_external))

## 4. Вспомогательные функции оптимизации

Для решения безусловных вспомогательных задач используется простой градиентный метод с backtracking line search.

Этот метод подбирает шаг так, чтобы значение функции уменьшалось и при необходимости не нарушались условия допустимости.

In [ ]:
def numerical_grad(func, x, h=1e-6):
    x = np.asarray(x, dtype=float)
    grad = np.zeros_like(x)

    for i in range(len(x)):
        x1 = x.copy()
        x2 = x.copy()
        x1[i] += h
        x2[i] -= h
        grad[i] = (func(x1) - func(x2)) / (2 * h)

    return grad


def gradient_descent(func, x_start, grad_func=None, max_inner_iter=1000, tol=1e-7, feasible_check=None):
    x = np.asarray(x_start, dtype=float).copy()
    history = [func(x)]

    for k in range(max_inner_iter):
        g = grad_func(x) if grad_func is not None else numerical_grad(func, x)

        if np.linalg.norm(g) < tol:
            break

        direction = -g
        alpha = 1.0
        current_value = func(x)

        success = False
        for _ in range(50):
            x_new = x + alpha * direction

            if feasible_check is not None and not feasible_check(x_new):
                alpha *= 0.5
                continue

            new_value = func(x_new)

            if np.isfinite(new_value) and new_value < current_value:
                success = True
                break

            alpha *= 0.5

        if not success:
            break

        x = x_new
        history.append(func(x))

        if len(history) > 2 and abs(history[-2] - history[-1]) < tol:
            break

    return x, history

## 5. Метод штрафных функций

Метод внешних штрафов преобразует задачу условной оптимизации в последовательность задач безусловной оптимизации.

К целевой функции добавляется штраф за нарушение ограничений:

$$
F(x, r) = f(x) + r \sum_j \max(0, g_j(x))^2
$$

Если точка нарушает ограничения, значение вспомогательной функции увеличивается. При росте параметра `r` алгоритм сильнее наказывает за выход из допустимой области.

In [ ]:
def penalty_function_method(x_start, r0=1.0, c=10.0, outer_iter=8):
    start_time = time.time()

    x = np.asarray(x_start, dtype=float).copy()
    full_history = []
    outer_data = []

    r = r0

    for outer in range(outer_iter):
        def F_penalty(z):
            g = constraints(z)
            penalty = np.sum(np.maximum(g, 0.0) ** 2)
            return f(z) + r * penalty

        x, hist = gradient_descent(F_penalty, x, max_inner_iter=1200, tol=1e-8)
        full_history.extend(hist)

        penalty_value = violation_norm(x)

        outer_data.append({
            "outer_iter": outer,
            "r": r,
            "x": x.copy(),
            "f": f(x),
            "violation": penalty_value
        })

        if penalty_value < eps:
            break

        r *= c

    elapsed = time.time() - start_time

    return {
        "Метод": "Штрафные функции",
        "x": x,
        "f": f(x),
        "violation": violation_norm(x),
        "iterations": len(full_history),
        "time": elapsed,
        "history": full_history,
        "outer_data": outer_data
    }


result_penalty = penalty_function_method(x0_external)

print("Метод штрафных функций")
print("x =", result_penalty["x"])
print("f(x) =", result_penalty["f"])
print("Нарушение ограничений =", result_penalty["violation"])
print("Время =", result_penalty["time"])

## 6. Метод барьерных функций

Метод барьерных функций применяется только из внутренней точки допустимой области.

Вспомогательная функция имеет вид:

$$
F(x, r) = f(x) - r \sum_j \ln(-g_j(x))
$$

Так как логарифм определен только при отрицательных значениях ограничений, алгоритм не позволяет точке выйти за границу допустимой области.

In [ ]:
def barrier_function_method(x_start, r0=1.0, c=5.0, outer_iter=8):
    start_time = time.time()

    x = np.asarray(x_start, dtype=float).copy()
    full_history = []
    outer_data = []

    r = r0

    def strictly_feasible(z):
        return np.all(constraints(z) < -1e-10)

    for outer in range(outer_iter):
        def F_barrier(z):
            g = constraints(z)
            if np.any(g >= 0):
                return np.inf
            return f(z) - r * np.sum(np.log(-g))

        x, hist = gradient_descent(
            F_barrier,
            x,
            max_inner_iter=1000,
            tol=1e-8,
            feasible_check=strictly_feasible
        )

        full_history.extend([val for val in hist if np.isfinite(val)])

        outer_data.append({
            "outer_iter": outer,
            "r": r,
            "x": x.copy(),
            "f": f(x),
            "violation": violation_norm(x)
        })

        if r < eps:
            break

        r /= c

    elapsed = time.time() - start_time

    return {
        "Метод": "Барьерные функции",
        "x": x,
        "f": f(x),
        "violation": violation_norm(x),
        "iterations": len(full_history),
        "time": elapsed,
        "history": full_history,
        "outer_data": outer_data
    }


result_barrier = barrier_function_method(x0_feasible)

print("Метод барьерных функций")
print("x =", result_barrier["x"])
print("f(x) =", result_barrier["f"])
print("Нарушение ограничений =", result_barrier["violation"])
print("Время =", result_barrier["time"])

## 7. Комбинированный метод штрафных и барьерных функций

Комбинированный метод использует две идеи одновременно:

1. Барьерную часть — чтобы удерживать точку внутри допустимой области.
2. Штрафную часть — чтобы наказывать за возможное нарушение ограничений.

В данной реализации метод стартует из допустимой внутренней точки.

In [ ]:
def combined_penalty_barrier_method(x_start, r_penalty0=1.0, r_barrier0=1.0, c=5.0, outer_iter=8):
    start_time = time.time()

    x = np.asarray(x_start, dtype=float).copy()
    full_history = []
    outer_data = []

    r_penalty = r_penalty0
    r_barrier = r_barrier0

    def strictly_feasible(z):
        return np.all(constraints(z) < -1e-10)

    for outer in range(outer_iter):
        def F_combined(z):
            g = constraints(z)
            penalty = np.sum(np.maximum(g, 0.0) ** 2)

            if np.any(g >= 0):
                barrier = 1e8 + 1e8 * penalty
            else:
                barrier = -r_barrier * np.sum(np.log(-g))

            return f(z) + r_penalty * penalty + barrier

        x, hist = gradient_descent(
            F_combined,
            x,
            max_inner_iter=1000,
            tol=1e-8,
            feasible_check=strictly_feasible
        )

        full_history.extend([val for val in hist if np.isfinite(val)])

        outer_data.append({
            "outer_iter": outer,
            "r_penalty": r_penalty,
            "r_barrier": r_barrier,
            "x": x.copy(),
            "f": f(x),
            "violation": violation_norm(x)
        })

        if r_barrier < eps and violation_norm(x) < eps:
            break

        r_penalty *= c
        r_barrier /= c

    elapsed = time.time() - start_time

    return {
        "Метод": "Комбинированный метод",
        "x": x,
        "f": f(x),
        "violation": violation_norm(x),
        "iterations": len(full_history),
        "time": elapsed,
        "history": full_history,
        "outer_data": outer_data
    }


result_combined = combined_penalty_barrier_method(x0_feasible)

print("Комбинированный метод")
print("x =", result_combined["x"])
print("f(x) =", result_combined["f"])
print("Нарушение ограничений =", result_combined["violation"])
print("Время =", result_combined["time"])

## 8. Метод модифицированных функций Лагранжа

Метод модифицированной функции Лагранжа добавляет к функции не только штраф, но и оценки множителей Лагранжа.

Для ограничений вида:

$$
g_j(x) \leq 0
$$

используется вспомогательная функция:

$$
L(x, \mu, r) = f(x) + \sum_j \mu_j g_j(x) + \frac{r}{2}\sum_j \max(0, g_j(x))^2
$$

После решения каждой вспомогательной задачи множители пересчитываются.

In [ ]:
def augmented_lagrangian_method(x_start, r0=10.0, c=2.0, outer_iter=12):
    start_time = time.time()

    x = np.asarray(x_start, dtype=float).copy()
    mu = np.zeros(4)
    r = r0

    full_history = []
    outer_data = []

    for outer in range(outer_iter):
        def L_aug(z):
            g = constraints(z)
            return f(z) + np.dot(mu, g) + 0.5 * r * np.sum(np.maximum(g, 0.0) ** 2)

        x, hist = gradient_descent(L_aug, x, max_inner_iter=1200, tol=1e-8)
        full_history.extend(hist)

        g = constraints(x)
        mu = np.maximum(0.0, mu + r * g)

        outer_data.append({
            "outer_iter": outer,
            "r": r,
            "x": x.copy(),
            "f": f(x),
            "violation": violation_norm(x),
            "mu": mu.copy()
        })

        if violation_norm(x) < eps:
            break

        r *= c

    elapsed = time.time() - start_time

    return {
        "Метод": "Модифицированная функция Лагранжа",
        "x": x,
        "f": f(x),
        "violation": violation_norm(x),
        "iterations": len(full_history),
        "time": elapsed,
        "history": full_history,
        "outer_data": outer_data,
        "mu": mu
    }


result_lagrange = augmented_lagrangian_method(x0_external)

print("Метод модифицированной функции Лагранжа")
print("x =", result_lagrange["x"])
print("f(x) =", result_lagrange["f"])
print("Нарушение ограничений =", result_lagrange["violation"])
print("Множители mu =", result_lagrange["mu"])
print("Время =", result_lagrange["time"])

## 9. Метод проекции градиента

Метод проекции градиента строит движение в направлении убывания функции, а затем проецирует новую точку на допустимое множество.

Для варианта 2 допустимое множество — это часть единичного шара в первом октанте. Поэтому после каждого шага выполняются две операции:

1. Отрицательные координаты заменяются нулями.
2. Если норма точки больше 1, точка проектируется на единичный шар.

In [ ]:
def project_to_feasible(x):
    x = np.asarray(x, dtype=float).copy()

    # Условие x_i >= 0
    x = np.maximum(x, 0.0)

    # Условие x1^2 + x2^2 + x3^2 <= 1
    norm = np.linalg.norm(x)
    if norm > 1.0:
        x = x / norm

    return x


def projected_gradient_method(x_start, max_pg_iter=5000, tol=1e-8):
    start_time = time.time()

    x = project_to_feasible(x_start)
    history = [f(x)]

    for k in range(max_pg_iter):
        g = grad_f(x)

        if np.linalg.norm(g) < eps:
            break

        alpha = 1.0
        current_value = f(x)

        success = False
        for _ in range(60):
            x_new = project_to_feasible(x - alpha * g)

            if f(x_new) <= current_value:
                success = True
                break

            alpha *= 0.5

        if not success:
            break

        history.append(f(x_new))

        if np.linalg.norm(x_new - x) < tol and abs(f(x_new) - f(x)) < tol:
            x = x_new
            break

        x = x_new

    elapsed = time.time() - start_time

    return {
        "Метод": "Проекция градиента",
        "x": x,
        "f": f(x),
        "violation": violation_norm(x),
        "iterations": len(history),
        "time": elapsed,
        "history": history
    }


result_projection = projected_gradient_method(x0_feasible)

print("Метод проекции градиента")
print("x =", result_projection["x"])
print("f(x) =", result_projection["f"])
print("Нарушение ограничений =", result_projection["violation"])
print("Время =", result_projection["time"])

## 10. Сравнение результатов

В данном блоке формируется итоговая таблица для сравнения методов.

В таблицу включены:

- найденная точка;
- значение целевой функции;
- значения ограничений;
- нарушение ограничений;
- количество итераций;
- время работы.

In [ ]:
results = [
    result_penalty,
    result_barrier,
    result_combined,
    result_lagrange,
    result_projection
]

summary = []

for r in results:
    x = r["x"]
    g = constraints(x)
    summary.append({
        "Метод": r["Метод"],
        "x1": x[0],
        "x2": x[1],
        "x3": x[2],
        "f(x)": r["f"],
        "g1": g[0],
        "g2": g[1],
        "g3": g[2],
        "g4": g[3],
        "Нарушение": r["violation"],
        "Итерации": r["iterations"],
        "Время, сек": r["time"],
        "Допустима": is_feasible(x)
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

## 11. График сходимости методов

На графике показано изменение значения целевой функции в процессе работы методов.

Для удобства используется логарифмическая шкала по оси Y.

In [ ]:
plt.figure(figsize=(10, 6))

for r in results:
    hist = np.array(r["history"], dtype=float)
    hist = hist[np.isfinite(hist)]
    if len(hist) > 0:
        plt.plot(hist - min(summary_df["f(x)"]) + 1e-8, label=r["Метод"])

plt.yscale("log")
plt.xlabel("Номер итерации")
plt.ylabel("f(x) - лучшее найденное значение")
plt.title("Сравнение сходимости методов условной оптимизации")
plt.legend()
plt.grid(True)
plt.show()

## 12. Сравнение методов по времени работы

Данный график показывает, сколько времени потребовалось каждому методу для получения результата.

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(summary_df["Метод"], summary_df["Время, сек"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Время, сек")
plt.title("Сравнение методов по времени работы")
plt.grid(axis="y")
plt.show()

## 13. Сравнение методов по количеству итераций

Данный график показывает количество итераций для каждого метода.

Количество итераций зависит от способа построения вспомогательной задачи и от особенностей выбранного метода.

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(summary_df["Метод"], summary_df["Итерации"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Количество итераций")
plt.title("Сравнение методов по количеству итераций")
plt.grid(axis="y")
plt.show()

## 14. Проверка решения на допустимость

Решение считается допустимым, если все ограничения имеют значения меньше или равные нулю:

$$
g_j(x) \leq 0
$$

В этом блоке дополнительно проверяется выполнение ограничений для каждой найденной точки.

In [ ]:
for r in results:
    x = r["x"]
    print("=" * 80)
    print(r["Метод"])
    print("x =", x)
    print("f(x) =", f(x))
    print("g(x) =", constraints(x))
    print("Нарушение ограничений =", violation_norm(x))
    print("Допустима ли точка:", is_feasible(x))

## 15. Стационарная точка и активные ограничения

Для задачи без ограничений минимум функции Розенброка находится в точке:

$$
(1, 1, 1)
$$

Однако эта точка не является допустимой, так как:

$$
1^2 + 1^2 + 1^2 - 1 = 2 > 0
$$

Следовательно, условный минимум должен находиться внутри допустимой области или на ее границе.

Для варианта 2 допустимая область ограничена единичным шаром и условиями неотрицательности координат.

In [ ]:
x_unconstrained = np.ones(3)

print("Безусловный минимум функции Розенброка:")
print("x =", x_unconstrained)
print("f(x) =", f(x_unconstrained))
print("g(x) =", constraints(x_unconstrained))
print("Допустим ли безусловный минимум:", is_feasible(x_unconstrained))

best_idx = summary_df["f(x)"].idxmin()
best_method = summary_df.loc[best_idx, "Метод"]
best_x = results[best_idx]["x"]

print("\nЛучшее найденное допустимое решение:")
print("Метод:", best_method)
print("x =", best_x)
print("f(x) =", f(best_x))
print("g(x) =", constraints(best_x))

## 16. Итоговый вывод

В ходе лабораторной работы были реализованы методы условной оптимизации:

1. Метод штрафных функций.
2. Метод барьерных функций.
3. Комбинированный метод штрафных и барьерных функций.
4. Метод модифицированной функции Лагранжа.
5. Метод проекции градиента.

Для варианта 2 была рассмотрена функция Розенброка с параметрами:

$$
a = 150, \quad b = 2, \quad f_0 = 100, \quad n = 3
$$

и ограничениями:

$$
x_1^2 + x_2^2 + x_3^2 \leq 1, \quad x_1 \geq 0, \quad x_2 \geq 0, \quad x_3 \geq 0
$$

Безусловная точка минимума функции Розенброка не входит в допустимую область, поэтому задача действительно является задачей условной оптимизации.

По итоговой таблице и графикам можно сравнить методы по точности, скорости сходимости и выполнению ограничений.

In [ ]:
print("Лабораторная работа №4 выполнена.")
print("Реализованы методы условной оптимизации для варианта 2.")
print("Построены таблицы и графики для сравнения результатов.")